# 멀티카메라 비디오 처리 시스템 테스트

`videoSCST`를 사용한 멀티카메라 비디오 처리 테스트 (최적화 버전)


In [1]:
import os, sys, time, glob
import torch

# 경로 추가 (프로젝트 루트)
sys.path.append("/workspace")

from MCMT_engine.SCST.video_SCST import videoSCST

## 1. 최적화된 Args 설정


In [ ]:
class Args:
    # 기존 트래킹 파라미터 유지
    track_thresh = 0.3
    match_thresh = 0.9
    track_buffer = 180
    mot20 = False

    # 배치/파이프라인
    batch_size   = 64      # GPU에 굵게 먹이기
    min_flush = 64        # 타임아웃을 쓰더라도 64장 모일 때만 플러시
    infer_timeout = 0.0
    cpu_workers  = 0       # 더 이상 사용 안 함(디코더가 스레드 내부에서 처리)
    chunk_sec    = 0.0     # 사용 안 함

    # 새 디코더 파라미터
    decode_threads  = 24    # CPU가 여유면 늘리고, 과하면 컨텍스트 스위칭 ↑
    prefetch_frames = 4096  # 여유 메모리 따라 128~512
    hwaccel         = "cuda"  # "cuda"/"nvdec" 가능 시 사용
    decode_target_size = None  # (1280,720) 처럼 지정하면 디코더에서 다운스케일


args = Args()

# GPU 메모리 최적화
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("CUDA를 사용할 수 없습니다. CPU 모드로 실행됩니다.")


GPU: NVIDIA RTX A6000


In [3]:
# 1) 도면 이미지 경로 (공통)
plan_path = "/workspace/assets/seocho/Seocho_plan_pts.png"

# 2) 카메라별 도면 기준점 (PLAN)
cam1_plan_pts = [
    (1431,1198), #측정안전통로
    (1505,1256), #노란깃발
    (1486,1068), #철근아래
    (1517,999),  #철근위
    (1505,972),  #나무적재왼쪽
    (1528,972),  #나무적재오른쪽
    (1601,1015), #범퍼
    (1664,1015), #초록깃발
    (1699,958),  #미니저수지왼쪽코너
    (1724,644),  #미니저수지위쪽코너
    (2026,449),  #출입구왼쪽
    (2032,532),  #출입구오른쪽
    (2016,615),  #컨테이너왼쪽
    (2019,654),  #컨테이너오른쪽
    (1917,939),  #미니저수지오른쪽
    (1908,625),  #미니저수지오른쪽윗코너
    (1219,1557)  #빨간꼬깔
]

cam2_plan_pts = [
    (1597,625),  #저수지 계단 아래
    (1629,619),  #저수지 계단 위
    (2023,451),  #출입구 왼쪽
    (2029,530),  #출입구 오른쪽
    (2015,615),  #컨테이너 왼쪽
    (2019,659),  #컨테이너 오른쪽
    (1372,907),  #저수지 꼭지
    (1375,952),  #꼭지 밑 안전통로
    (1505,973),  #나무적재왼쪽
    (1525,971),  #나무적재오른쪽
    (1520,998),  #철근위
    (1485,1068), #철근아래
    (1330,1177), #안전통로입구
    (1431,1198), #측정안전통로
    (1594,1176), #범퍼
    (1504,1253), #노란깃발
    (1698,960),  #작은저수지왼쪽코너
    (1915,942),  #작은저수지오른쪽코너
]

cam3_plan_pts = [
    (1597,627),  #저수지계단아래
    (1635,619),  #저수지계단위
    (1505,1256), #노란깃발
    (1695,958),  #미니저수지왼쪽아래
    (1663,1017), #초록깃발
    (1036,378),  #미니저수지오른쪽아래
    (1724,643),  #미니저수지왼쪽위
    (1910,630),  #미니저수지오른쪽 위
    (1597,1174), #범퍼
    (1616,903),  #파란깃발
    (2025,453),  #출입구왼쪽
    (2028,529),  #출입구오른쪽
    (2015,614),  #컨테이너왼쪽
    (2019,662),  #컨테이너 오른쪽
    (769,587),   #파란포장
    (1505,973),  #나무적재왼쪽
    (1531,967),  #나무적재오른쪽
    (1477,1088), #철근아래
    (1454,411),  #직원휴게실왼쪽
    (1568,377),  #직원휴게실오른쪽
]

print("✅ 캘리브레이션 포인트 설정 완료")


✅ 캘리브레이션 포인트 설정 완료


In [4]:
# 3) 카메라별 영상 좌표 (CCTV)
cam1_pts = [
    (342,661),  #측정안전통로
    (733,657),  #노란깃발
    (266,567),  #철근아래
    (230,526),  #철근위
    (129,519),  #나무적재왼쪽
    (188,514),  #나무적재오른쪽
    (811,603),  #범퍼
    (612,525),  #초록깃발
    (600,516),  #미니저수지왼쪽코너
    (177,403),  #미니저수지위쪽코너
    (420,382),  #출입구왼쪽
    (506,395),  #출입구오른쪽
    (577,421),  #컨테이너왼쪽
    (623,416),  #컨테이너오른쪽
    (979,493),  #미니저수지오른쪽
    (451,418),  #미니저수지오른쪽윗코너
    (961,717),  #빨간꼬깔
]

cam2_pts = [
    (2,462),    #저수지 계단아래
    (25,420),   #저수지 계단 위
    (107,391),  #출입구 왼쪽
    (212,399),  #출입구 오른쪽
    (309,409),  #컨테이너 왼쪽
    (360,417),  #컨테이너 오른쪽
    (313,542),  #저수지 꼭지
    (319,564),  #꼭지 밑 안전통로
    (561,503),  #나무적재왼쪽
    (580,496),  #나무적재오른쪽
    (633,506),  #철근위
    (782,530),  #철근 아래
    (966,608),  #안전통로입구
    (1055,571), #측정안전통로
    (1051,531), #범퍼
    (1206,555), #노란깃발
    (674,485),  #작은저수지왼쪽코너
    (728,459),  #작은저수지오른쪽코너
]

cam3_pts = [
    (267,350),  #저수지계단아래
    (310,305),  #저수지계단위
    (104,706),  #노란깃발
    (649,407),  #미니저수지왼쪽아래
    (664,422),  #초록깃발
    (1036,378), #미니저수지오른쪽아래
    (448,309),  #미니저수지왼쪽위
    (692,318),  #미니저수지오른쪽 위
    (539,549),  #범퍼
    (372,362),  #파란깃발
    (711,299),  #출입구왼쪽
    (763,306),  #출입구오른쪽
    (798,311),  #컨테이너왼쪽
    (827,319),  #컨테이너 오른쪽
    (769,587),  #파란포장
    (119,419),  #나무적재왼쪽
    (176,408),  #나무적재오른쪽
    (60,471),   #철근아래
    (63,301),   #직원휴게실왼쪽
    (192,291),  #직원휴게실오른쪽
]

print("✅ CCTV 좌표 설정 완료")


✅ CCTV 좌표 설정 완료


## 2. 비디오 파일 확인 및 공유 모델 초기화


In [5]:
# 4) 비디오 파일
video_paths = [
    "/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #1.mp4",
    "/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #2.mp4",
    "/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #3.mp4",
]

existing_videos = []
for i, video_path in enumerate(video_paths):
    if os.path.exists(video_path):
        existing_videos.append(video_path)
        print(f"Camera {i+1}: {video_path}")
    else:
        print(f"Camera {i+1}: {video_path} (파일 없음)")

if len(existing_videos) < 3:
    raise FileNotFoundError(f"비디오 3개가 모두 필요합니다. 현재 {len(existing_videos)}개만 확인됨.")

print(f"\n총 {len(existing_videos)}개 비디오 파일 확인됨")


# 5) 결과 저장 디렉토리
os.makedirs("/workspace/results", exist_ok=True)
print("결과 저장 디렉토리 준비 완료")


Camera 1: /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #1.mp4
Camera 2: /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #2.mp4
Camera 3: /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #3.mp4

총 3개 비디오 파일 확인됨
결과 저장 디렉토리 준비 완료


In [6]:
# 6) videoSCST 초기화
print("\nvideoSCST 초기화 중...")
scst = videoSCST(
    plan_path=plan_path,
    args=args,
    det_models=["ultra_best"],     # 실제 모델 키 확인 필요
    det_device="cuda:0",
    det_threshold=0.0,
    det_use_async=True,
    det_max_workers=1,
)
print("videoSCST 초기화 완료")


videoSCST 초기화 중...
[SCST] __init__: plan_path=/workspace/assets/seocho/Seocho_plan_pts.png
[SCST] __init__: create DetectionAPI(device=cuda:0, models=['ultra_best'], async=True)
DetectionAPI activate
[DEBUG] VehicleDetector loaded with Ultralytics YOLO on cuda:0
[SCST] __init__: create TrackerAPI
TrackingAPI activated...
[TrackerAPI] __init__: decode_threads=16, prefetch=4096, hwaccel=cuda, target_size=None, batch_size=64
videoSCST 초기화 완료


In [7]:
# 7) 각 카메라별 처리
start_time = time.time()
results = []

print("캘리브레이션 포인트 설정 완료")

캘리브레이션 포인트 설정 완료


In [ ]:
# Camera 1
print("\nCamera 1 처리 시작...")
cam_start = time.time()
result1 = scst.track_and_save(
    video_path=existing_videos[0],
    cam_pts=cam1_pts,
    plan_pts=cam1_plan_pts,
    plan_img_path=plan_path,
    camera_save_path="/workspace/results/tracking_result1new.mp4",
    plan_save_path="/workspace/results/plan_result1.mp4",
    cam_trail_len=30,
    plan_stride=1,          # 필요 시 2~3으로 올려 저장 부하/용량 감소
    inplace_clear=True      # 메모리 즉시 해제
)
results.append(result1)
cam_time = time.time() - cam_start
print(f"Camera 1 처리 완료 ({cam_time:.2f}초, {len(result1)} 프레임)")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Camera 1 처리 시작...
[SCST] calibrate: start
[SCST] _ensure_projector: assign plan image (/workspace/assets/seocho/Seocho_plan_pts.png)
[SCST] calibrate: create PlanProjector + initial H fit
[SCST] calibrate: done |H|=4606.8280 time=0.202s
[SCST] tracking: begin (decode via iter_frames_parallel, detect+associate inside TrackerCore)
[SCST] tracking: with camera visualization → /workspace/results/tracking_result1new.mp4
TrackingAPI : tracking video
[TrackerAPI] encode(camera): prepare writer for /workspace/results/tracking_result1new.mp4
[TrackerAPI] _create_writer: mp4v OK → /workspace/results/tracking_result1new.mp4
[TrackerAPI] encode(camera): writer ready path=/workspace/results/tracking_result1new.mp4, fps=30.00, size=(1920x1080)
[TrackerAPI] decode(streaming): start (threads=16, prefetch=4096, hwaccel=cuda, target_size=None)
[TrackerAPI] detect+track+encode(camera): start
[decoder] open(PyAV): threads=16, hwaccel=cuda
DetectionAPI : Batch Detection..
DetectionAPI : Batch Detection..


In [ ]:
# Camera 2
print("\nCamera 2 처리 시작...")
cam_start = time.time()
result2 = scst.track_and_save(
    video_path=existing_videos[1],
    cam_pts=cam2_pts,
    plan_pts=cam2_plan_pts,
    plan_img_path=plan_path,
    camera_save_path="/workspace/results/tracking_result2new.mp4",
    plan_save_path="/workspace/results/plan_result2.mp4",
    cam_trail_len=30,
    plan_stride=1,
    inplace_clear=True
)
results.append(result2)
cam_time = time.time() - cam_start
print(f"Camera 2 처리 완료 ({cam_time:.2f}초, {len(result2)} 프레임)")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Camera 3
print("\nCamera 3 처리 시작...")
cam_start = time.time()
result3 = scst.track_and_save(
    video_path=existing_videos[2],
    cam_pts=cam3_pts,
    plan_pts=cam3_plan_pts,
    plan_img_path=plan_path,
    camera_save_path="/workspace/results/tracking_result3new.mp4",
    plan_save_path="/workspace/results/plan_result3.mp4",
    cam_trail_len=30,
    plan_stride=1,
    inplace_clear=True
)
results.append(result3)
cam_time = time.time() - cam_start
print(f"Camera 3 처리 완료 ({cam_time:.2f}초, {len(result3)} 프레임)")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 8) 요약
total_time = time.time() - start_time
total_frames = sum(len(r) for r in results)
print(f"\n전체 완료: {total_time:.2f}초, 총 {total_frames} 프레임")

## 4. 결과 요약 및 리소스 정리


In [ ]:
# 9) 결과 요약 및 파일 확인
print("\n모든 카메라 처리 완료!")
print(f"총 처리 시간: {total_time:.2f}초")
print(f"총 처리 프레임: {total_frames:,}개")
print(f"평균 처리 속도: {total_frames/total_time:.1f} FPS")
print("결과 저장 위치: /workspace/results/")

result_files = glob.glob("/workspace/results/*.mp4")
print(f"\n생성된 결과 파일 ({len(result_files)}개):")
for fp in sorted(result_files):
    size_mb = os.path.getsize(fp) / (1024*1024)
    print(f"  - {os.path.basename(fp)}  ({size_mb:.1f} MB)")

print("\n완료")